# GPU Pilot (headless) — One-Command Hospital on Kaggle T4 x2

**Automated headless runbook** (pushed via Kaggle API; no human input cells):
- repo snapshot restored from private dataset `aminurhakim/och-snapshot` — **no GitHub PAT inside this kernel**
- vLLM serves **BioMistral-7B (official AWQ 4-bit → BnB.4 fallback → fp16 last resort)** — all public HF repos, no HF token
- native stack in GPU shape (`LLM_URL` override, v0.6.2), guided decoding active
- gates: test-unit / test-node / test-integration-native / eval (seed) / eval-full (208) / LIVE strict (LIVE_MODE=gpu) / loadtest (pilot-grade) / audit-verify
- evidence pack (`gpu_pilot_evidence_<stamp>.tar.gz` + sha256 + `HOST_manifest.json` + `gate_summary.json`) lands in `/kaggle/working` = notebook output

**This is NOT the Gate-5 sign-off run** — free shared VM; latency evidence is pilot-grade only
(see `docs/gpu_pilot_plan.md` §4). Tier-1 success = trap refusal ≥90% AND citation validity ≥80% AND red-team floor held.

Run config: Accelerator **GPU T4 ×2** (`machine_shape: NvidiaTeslaT4`), Internet **On**, private kernel.


In [ ]:
# CELL 1 — Host introspection: confirm GPU is actually attached before anything else
import subprocess, os, platform
print("== HOST INTROSPECTION ==")
nvid = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(nvid.stdout if nvid.returncode == 0 else "!!! NO GPU VISIBLE — attach the GPU accelerator (Kaggle: Settings>Accelerator / Colab: Runtime>Change runtime type) and rerun from scratch !!!")
print("python:", platform.python_version(), "| cpus:", os.cpu_count())
print("disk free (GB):", round(__import__('shutil').disk_usage('/').free/1e9, 1))
try:
    print(open('/proc/meminfo').readline().strip())
except Exception:
    pass
assert nvid.returncode == 0, "No GPU — aborting before wasting quota."


In [ ]:
# CELL 2 — Restore repo snapshot from the private Kaggle dataset (headless; NO tokens in this kernel)
import tarfile, os, glob, shutil, hashlib, subprocess

BASE = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"
SNAP = None
for pat in ["/kaggle/input/och-snapshot/och-snapshot.tar.gz",
            "/kaggle/input/*/och-snapshot.tar.gz",
            "/content/och-snapshot.tar.gz"]:
    hits = glob.glob(pat)
    if hits:
        SNAP = hits[0]; break
assert SNAP, "och-snapshot.tar.gz not found under /kaggle/input — attach dataset aminurhakim/och-snapshot and rerun"

WORK = os.path.join(BASE, "repo")
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)
snap_sha = hashlib.sha256(open(SNAP, "rb").read()).hexdigest()
with tarfile.open(SNAP) as t:
    t.extractall(WORK)

REPO_DIR = os.path.join(WORK, "download", "one-command-hospital")
assert os.path.isdir(REPO_DIR), f"unexpected snapshot layout: {os.listdir(WORK)}"
os.chdir(REPO_DIR)
os.environ["OCH_SNAPSHOT_SHA256"] = snap_sha
print("repo restored ->", os.getcwd())
print("snapshot sha256:", snap_sha)
print("top-level:", sorted(os.listdir("."))[:24])


In [ ]:
# CELL 3 — Interface discovery (RUNBOOK SAFETY): surface the true M17-era interface before wiring anything
# The notebook was authored against: ports 8100-8103 (prod-shape) / 8210-8213 (native test), LIVE_MODE=gpu,
# native runner tools/native_stack.sh, make bootstrap-native, evidence under eval/evidence/.
# If ANY of that has drifted between M7-M17, this cell SHOWS it. Read the output; adjust the cells below or STOP.
import subprocess, os, glob

def show(path, n=90):
    print(f"===== {path} {'(MISSING)' if not os.path.exists(path) else ''} =====")
    if os.path.exists(path):
        print("\n".join(open(path, errors='replace').read().splitlines()[:n]))

for p in ["STATE.md", "Makefile", "tools/native_stack.sh", "tools/bootstrap.sh",
          "deploy/vllm/README.md", "docs/gpu_pilot_plan.md"]:
    show(p, 70)

print("===== grep strict-gate + generator wiring =====")
subprocess.run("grep -rn 'LIVE_MODE' tests/live docs/world_class_bar.md | head -12 || true", shell=True)
subprocess.run("grep -nE 'LLM_URL|LLM_BASE_URL' tools/native_stack.sh services/guideline-rag/app/main.py | head -12 || true", shell=True)
subprocess.run("grep -nE 'GUIDED_JSON_FIELD|GUIDED_DECODING' deploy/vllm/README.md | head -8 || true", shell=True)
print("===== make targets (help) =====")
subprocess.run("make help 2>/dev/null || true", shell=True)
print("===== eval/evidence shape =====")
print(glob.glob("eval/evidence/*")[:20] or "(no evidence dir yet — expected)")
print("\n>>> VERIFY: LLM_URL override present in native_stack.sh (v0.6.2); LIVE_MODE=gpu live tier in tests/live;")
print(">>> gate targets = test-unit / test-node / test-integration-native / eval / eval-full / loadtest / audit-verify.")
print(">>> If anything differs from the cells below, FIX THE CELLS (not the repo) or STOP and reconcile with STATE.md.")


In [ ]:
# CELL 4 — Dependencies: eval + tests requirements + vLLM (pip; notebook VMs have no Docker daemon -> native path)
import subprocess, sys, os
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], capture_output=True, text=True)
for req in ["eval/requirements.txt", "tests/requirements.txt"]:
    if os.path.exists(req):
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req], capture_output=True, text=True)
        print(req, "->", (r.stdout or r.stderr)[-400:])
    else:
        print("!!", req, "not found — discovery cell output above should explain the tree shape")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
print("vllm install tail:", (r.stdout or r.stderr)[-600:])
r = subprocess.run([sys.executable, "-c", "import vllm; print('vllm OK', vllm.__version__)"], capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, "vLLM install failed"


In [ ]:
# CELL 5 — Launch vLLM: BioMistral-7B 4-bit on :8099 (official public AWQ -> BnB.4 -> fp16 ladder)
# No HF token needed: every candidate is a PUBLIC, non-gated HF repo (BioMistral org).
import subprocess, time, os, signal, sys
import requests

PORT = 8099
# (model_id, extra vllm flags) — int4 first (fits free T4 16GB), fp16 last resort at reduced ctx
CANDIDATES = [
    ("BioMistral/BioMistral-7B-AWQ-QGS128-W4-GEMM",
     ["--max-model-len", "4096", "--gpu-memory-utilization", "0.90"]),
    ("BioMistral/BioMistral-7B-BnB.4",
     ["--quantization", "bitsandbytes", "--load-format", "bitsandbytes",
      "--max-model-len", "4096", "--gpu-memory-utilization", "0.90"]),
    ("BioMistral/BioMistral-7B",
     ["--max-model-len", "2048", "--gpu-memory-utilization", "0.95"]),
]
proc, model_used, ok = None, None, False

for m, extra in CANDIDATES:
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
           "--model", m, "--port", str(PORT), "--enforce-eager"] + extra
    print("launching", m, "(cold download ~5GB — expect up to 20 min)...")
    proc = subprocess.Popen(cmd, stdout=open(f"/tmp/vllm_{PORT}.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(150):  # 150 x 10s = 25 min per candidate
        time.sleep(10)
        try:
            if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
                ok = True; break
        except Exception:
            pass
        if proc.poll() is not None:
            break
    if ok:
        model_used = m; break
    print("failed:", m, "— tail of log:")
    print(open(f"/tmp/vllm_{PORT}.log").read()[-1200:])
    try: proc.send_signal(signal.SIGTERM)
    except Exception: pass
    proc = None

assert ok, f"vLLM failed for all candidates — inspect /tmp/vllm_{PORT}.log (OOM? quota? kernel mismatch?)"
print("vLLM healthy:", model_used, "on :", PORT)
os.environ["GENERATOR_BASE_URL"] = f"http://127.0.0.1:{PORT}/v1"
os.environ["GENERATOR_MODEL"] = model_used
s = requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions",
                  json={"model": model_used, "messages": [{"role": "user", "content": "Say OK"}], "max_tokens": 8}, timeout=60)
print("smoke:", s.status_code, s.json()["choices"][0]["message"]["content"][:40] if s.status_code == 200 else s.text[:200])


In [ ]:
# CELL 6 — Start NATIVE stack in GPU shape + health-wait (prod ports 8100-8103)
# v0.6.2: native_stack.sh up-prod accepts LLM_URL override (was hardcoded to a dead URL = mock mode).
# GUIDED_DECODING=1 is the service default; GUIDED_JSON_FIELD per deploy/vllm/README.md — latest vLLM
# builds group guided_json under structured_outputs, so we pre-set the new field name (harmless on old builds).
import subprocess, os, time
import requests

os.environ["LLM_URL"] = f"http://127.0.0.1:{PORT}/v1"
os.environ.setdefault("GUIDED_DECODING", "1")
os.environ.setdefault("GUIDED_JSON_FIELD", "structured_outputs")  # flip to guided_json only if vLLM warns the other way

r = subprocess.run(["bash", "tools/native_stack.sh", "up-prod"], capture_output=True, text=True)
print((r.stdout or r.stderr)[-3000:])

print("== health poll (90s budget) ==")
healthy = {}
for attempt in range(18):
    healthy.clear()
    for port in (8100, 8101, 8102, 8103):
        try:
            resp = requests.get(f"http://127.0.0.1:{port}/health", timeout=2)
            healthy[port] = resp.status_code
        except Exception:
            healthy[port] = "down"
    if all(v == 200 for v in healthy.values()):
        break
    time.sleep(5)
print("health:", healthy)
assert all(v == 200 for v in healthy.values()), "Stack not healthy on 8100-8103 — check the log dir printed by native_stack.sh above."
# generator really wired? /answer must NOT be extractive-mock: ask and eyeball the answer shape
q = requests.post("http://127.0.0.1:8101/answer", json={"question": "What is the bridging plan for warfarin before surgery?"}, timeout=120)
print("generator smoke:", q.status_code, str(q.json())[:300])


In [ ]:
# CELL 7 — Gates: unit / node / integration-native / eval(seed) / eval-full(208) / LIVE strict / loadtest(pilot) / audit
import subprocess, os, json, time

RAG = "http://127.0.0.1:8101"
GATES = []

def run(cmd, timeout=3600):
    print()
    print(f">>> {cmd}")
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    print(r.stdout[-4500:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-2000:])
    print(f"--- exit: {r.returncode} ({time.time()-t0:.0f}s) ---")
    GATES.append({"cmd": cmd.split()[0] if cmd.startswith("make") else cmd.split(" ")[0],
                  "exit": r.returncode, "seconds": round(time.time() - t0, 1)})
    return r

run("make test-unit")                       # pytest unit tier
run("make test-node")                       # audit chain, redaction, metrics
run("make test-integration-native")         # golden path / fail-closed / PHI on mock test stack (8210-8213)
run(f"RAG_URL={RAG} make eval")             # seed eval, full mode (citations checked vs live stack)
run(f"RAG_URL={RAG} make eval-full")        # 208-question eval vs live GPU-backed stack
run(f"RAG_URL={RAG} LIVE_MODE=gpu python -m pytest tests/live -m live -v")  # STRICT: trap refusal >=90%, citation validity >=80%, red-team floor
run("make loadtest LOCUST_USERS=20 LOCUST_RUN_TIME=2m")  # pilot-grade latency ONLY (shared VM)
run("make audit-verify")                    # hash-chained audit log walk

base = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"
summary = {"gates": GATES,
           "tier1_live_exit": next((g["exit"] for g in GATES if "pytest" in g["cmd"]), None),
           "generator_model": os.environ.get("GENERATOR_MODEL"),
           "snapshot_sha256": os.environ.get("OCH_SNAPSHOT_SHA256"),
           "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
json.dump(summary, open(os.path.join(base, "gate_summary.json"), "w"), indent=2)
print(json.dumps(summary, indent=2))

print("\n>>> READ tests/live OUTPUT — Tier-1 success = trap refusal >= 90% AND citation validity >= 80% AND red-team floor held")
print(">>> Latency numbers are PILOT-GRADE ONLY (shared VM). Gate-5 sign-off loadtest runs on local/24GB or a ~$2-5 paid spot.")


In [ ]:
# CELL 8 — Evidence harvest: HOST manifest + logs + tarball + sha256 (Kaggle: /kaggle/working = output)
import subprocess, os, json, hashlib, tarfile, time, shutil, glob

stamp = time.strftime("%Y%m%d_%H%M%S")
gpu_q = subprocess.run(["bash", "-lc", "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"],
                       capture_output=True, text=True).stdout.strip()
host = {"provider": "kaggle" if os.path.isdir("/kaggle") else "colab" if os.path.isdir("/content") else "unknown",
        "gpu": gpu_q, "shared_vm": True, "timestamp_utc": stamp,
        "model": os.environ.get("GENERATOR_MODEL"),
        "snapshot_sha256": os.environ.get("OCH_SNAPSHOT_SHA256")}
manifest = {"host": host,
            "caveat": "free shared-VM — pilot-grade evidence, not Gate-5 sign-off",
            "gate5_signoff": False}
os.makedirs("eval/evidence", exist_ok=True)
json.dump(manifest, open("eval/evidence/HOST_manifest.json", "w"), indent=2)

# carry run logs into the evidence dir (best-effort)
base = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"
try:
    if os.path.exists(f"/tmp/vllm_{PORT}.log"):
        open("eval/evidence/vllm_8099_tail.log", "w").write(open(f"/tmp/vllm_{PORT}.log").read()[-20000:])
except Exception as e:
    print("vllm log copy skipped:", e)
try:
    for lg in glob.glob("/tmp/och-native-stack/logs/*"):
        shutil.copy(lg, "eval/evidence/native_" + os.path.basename(lg))
except Exception as e:
    print("stack log copy skipped:", e)
try:
    shutil.copy(os.path.join(base, "gate_summary.json"), "eval/evidence/gate_summary.json")
except Exception as e:
    print("gate summary copy skipped:", e)

out = os.path.join(base, f"gpu_pilot_evidence_{stamp}.tar.gz")
with tarfile.open(out, "w:gz") as t:
    t.add("eval/evidence", arcname="eval/evidence")
digest = hashlib.sha256(open(out, "rb").read()).hexdigest()
print("evidence tarball:", out)
print("sha256:", digest)
print(json.dumps(manifest, indent=2))

try:
    from google.colab import drive  # Colab only
    drive.mount("/content/drive")
    shutil.copy(out, "/content/drive/MyDrive/")
    print("copied to Google Drive: MyDrive/" + os.path.basename(out))
except Exception as e:
    print("(Drive copy skipped:", type(e).__name__, "— on Kaggle the tarball is in /kaggle/working notebook output)")


In [ ]:
# CELL 9 — Teardown (stop vLLM, bring stack down, confirm ports freed)
import subprocess, signal, os
try:
    proc.send_signal(signal.SIGTERM)  # vLLM
    print("vLLM SIGTERM sent")
except Exception as e:
    print("vLLM stop:", type(e).__name__)
r = subprocess.run(["bash", "tools/native_stack.sh", "down"], capture_output=True, text=True)
print((r.stdout or r.stderr)[-1200:])
subprocess.run("pkill -f vllm || true", shell=True)
subprocess.run("pkill -f uvicorn || true", shell=True)
import requests
for port in (8100, 8101, 8102, 8103, 8099):
    try:
        requests.get(f"http://127.0.0.1:{port}/health", timeout=1)
        print(port, "STILL UP — kill manually")
    except Exception:
        print(port, "down")
print("teardown complete — remember: tarball + sha256 + HOST manifest are the pilot evidence pack")
